# Chat With Your Trained Adapter (Colab)

Use this notebook to run inference quickly on Colab GPU using your downloaded `adapter.zip`.

In [ ]:
!pip -q install transformers peft accelerate bitsandbytes

In [ ]:
import os
import zipfile
from pathlib import Path
from google.colab import files

print('Upload adapter.zip now')
uploaded = files.upload()
zip_name = next(iter(uploaded.keys()))
print('Uploaded:', zip_name)

extract_dir = Path('/content/adapter_extracted')
extract_dir.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(zip_name, 'r') as archive:
    archive.extractall(extract_dir)

if (extract_dir / 'outputs' / 'adapter').exists():
    ADAPTER_DIR = extract_dir / 'outputs' / 'adapter'
else:
    ADAPTER_DIR = extract_dir

print('Adapter dir:', ADAPTER_DIR)

In [ ]:
import torch
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

BASE_MODEL = 'Qwen/Qwen2.5-3B-Instruct'
USE_4BIT = True

tokenizer = AutoTokenizer.from_pretrained(str(ADAPTER_DIR), trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

quant_config = None
if USE_4BIT:
    quant_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type='nf4',
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
    )

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    device_map='auto',
    quantization_config=quant_config,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    trust_remote_code=True,
)
model = PeftModel.from_pretrained(model, str(ADAPTER_DIR))
model.eval()
print('Model loaded on:', 'GPU' if torch.cuda.is_available() else 'CPU')

In [ ]:
SYSTEM_PROMPT = 'Reply in the user texting style: concise, casual, and context-aware.'
history = []

def chat_once(user_text, max_new_tokens=120, temperature=0.8, top_p=0.9):
    messages = []
    if SYSTEM_PROMPT:
        messages.append({'role': 'system', 'content': SYSTEM_PROMPT})
    messages.extend(history)
    messages.append({'role': 'user', 'content': user_text})

    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors='pt').to(model.device)

    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=temperature,
            top_p=top_p,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.pad_token_id,
        )

    new_tokens = output[0][inputs['input_ids'].shape[-1]:]
    reply = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()
    history.append({'role': 'user', 'content': user_text})
    history.append({'role': 'assistant', 'content': reply})
    return reply

In [ ]:
user_text = 'Hey what are you up to?'
print(chat_once(user_text))

In [ ]:
# Rerun this cell for next messages
next_text = input('you> ')
print('bot>', chat_once(next_text))